# CLYRO — бесплатное QLoRA-дообучение Qwen3-4B в Google Colab

Этот ноутбук сохраняет датасет, чекпоинты и итоговый адаптер на Google Drive. Если сессия Colab оборвётся, запусти ячейки снова: скрипт продолжит обучение с последнего чекпоинта.

Нужен GPU T4 (если доступен) и минимум 100 проверенных примеров. Не запускай обучение на двух тестовых строках.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = '/content/drive/MyDrive/CLYRO'
DATA = f'{BASE}/train.jsonl'
OUT = f'{BASE}/checkpoints/run1'
os.makedirs(BASE, exist_ok=True)
os.makedirs(OUT, exist_ok=True)
print('Датасет найден:', os.path.exists(DATA), DATA)
print('Чекпоинты будут храниться:', OUT)

In [ ]:
!git clone https://github.com/pdvisual/project-ai-rebuild.git /content/project-ai-rebuild || (cd /content/project-ai-rebuild && git pull)
%cd /content/project-ai-rebuild
!pip install -q -r training/requirements.txt

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('Включи GPU: Runtime → Change runtime type → T4 GPU')

## Добавь датасет

Загрузи свой проверенный `train.jsonl` в папку Google Drive `MyDrive/CLYRO/`. В каждой строке должен быть JSON-объект с `messages`, содержащими сообщения `user` и `assistant`. Нужно минимум 100 качественных примеров для обычного запуска. Не добавляй пароли, API-ключи или приватные переписки.

In [ ]:
import json
if not os.path.exists(DATA):
    raise FileNotFoundError(f'Сначала положи train.jsonl в Google Drive: {DATA}')
with open(DATA, encoding='utf-8') as f:
    rows = [json.loads(line) for line in f if line.strip()]
print('Примеров:', len(rows))
if len(rows) < 100:
    raise ValueError('Пока меньше 100 примеров. Сначала расширь и проверь датасет; обучение не запущено.')
print('Проверка количества пройдена.')

In [ ]:
!python training/train_lora.py --data "$DATA" --output "$OUT" --epochs 1 --save-steps 25 --resume-from-checkpoint auto

## После завершения

Адаптер и токенизатор будут в `MyDrive/CLYRO/checkpoints/run1`. Это не отдельная модель и пока не подключено к рабочему чату CLYRO: сначала нужно отдельно проверить качество и подготовить inference-сервис.